## Lab 6: Classification fine-tuning
**Group:** Phillip Graf, Konstantin Schmidt, Fabian Holländer

In this lab, we import our classification dataset, perform necessary preprocessing steps and the train the OpenAI Gpt2 model, such that it can label reciepes into 9 different categories. The cool thing is that multilabeling is possible. That means if a reciepe is a bakery and vegetable the model classify it as it is.

The 9 different categories are:
.  
.  
TODO Beschreibung vollenden...  
.  
.  



For learning and testing purposes we import a reduced classification dataset of 100.000 annotated reciepes.
https://syncandshare.lrz.de/dl/fiWvy4z3fJ2sRSLMTdQiay/reduced_classification_dataset_100k.csv

*Optional* the full dataset containing 2 million annotated reciepes and can be used as input for larger-scale training.
https://syncandshare.lrz.de/dl/fiGzZTKfHqLLLbK2e6Uop9/full_classification_dataset.csv

## Importieren notwendiger Python Bibliotheken

In [34]:
import pandas as pd
import os
import re
import tiktoken
import torch
from torch.utils.data import Dataset, DataLoader
from importlib.metadata import version
from utils.gpt_download import download_and_load_gpt2
from utils.gpt import GPTModel, load_weights_into_gpt
from utils.gpt import (
    generate_text_simple,
    text_to_token_ids,
    token_ids_to_text
)
torch.manual_seed(123)

pkgs = ["matplotlib",  # Plotting library
        "numpy",       # PyTorch & TensorFlow dependency
        "tiktoken",    # Tokenizer
        "torch",       # Deep learning library
        "tensorflow",  # For OpenAI's pretrained weights
        "pandas"       # Dataset loading
       ]
for p in pkgs:
    print(f"{p} version: {version(p)}")

tokenizer = tiktoken.get_encoding("gpt2")

matplotlib version: 3.10.8
numpy version: 2.0.2
tiktoken version: 0.9.0
torch version: 2.10.0
tensorflow version: 2.21.0
pandas version: 2.3.3


## Laden und vorbereiten des Datensatzes

Der Datensatz hat eigentlich 5 Spalten... wir bereiten diese so auf, dass wir nur noch ein Text haben und 2 Classifzierungsspalten. Da im späteren Verlauf ein LLM auch das Gesamte Rezept als Input nimmt und nicht aufgeteilt in verschiedene Spalten.

Wir lassen uns ausgeben welche Label wir haben und wieviele Zeilen pro Label und bringen den Datensatz danach ins Gleichgewicht, damit ein label, dass Training nicht verzerrt oder ein Label nicht untereprsäentiert ist. Deshalb undersampeln wir hier.

Inklusive auf splitten in Traingings und Validierungs und test datensatz

### Design-Entscheidung: Textstrukturierung & Tokenisierung

**Überlegung:**
Ursprünglich wurde in Erwägung gezogen, die Abschnitte der Rezepte (Titel, Zutaten, Anleitung) über explizite Steuerungstokens (*Special Control Tokens* wie `<|startofingredients|>`) zu trennen. Dies ist architektonisch sauber, da das Modell so die semantischen Grenzen im Text exakt erlernen kann.

**Gewählte Lösung & Begründung:**
Wir haben uns letztlich gegen eigene Special Tokens und **für die Strukturierung mittels klassischer Zeilenumbrüche (`\n`)** entschieden. 

* **Problem bei Custom Tokens:** Da wir ein **vortrainiertes GPT-2-Modell** nutzen, würde das Hinzufügen neuer Tokens das originale Vokabular verändern. Die Gewichte des Embedding-Layers würden nicht mehr passen (`Shape Mismatch`), und das Modell müsste für die neuen Tokens komplett neue Vektoren von null auf lernen.
* **Vorteil von `\n`:** GPT-2 wurde auf riesigen Textmengen trainiert und hat bereits verinnerlicht, dass Zeilenumbrüche (`\n`) gefolgt von Schlüsselwörtern wie `Ingredients:` oder `Directions:` Abschnitte strukturieren. Wir nutzen dieses vortrainierte Wissen (*Prior Knowledge*) aus, um das Fine-Tuning stabiler und effizienter zu gestalten.

In [ ]:
# Reduced dataset with 100k rows for testing
cloud_url = "https://syncandshare.lrz.de/dl/fiWvy4z3fJ2sRSLMTdQiay/reduced_classification_dataset_100k.csv"
# Uncomment the following line to use the full annotated dataset
# cloud_url = "https://syncandshare.lrz.de/dl/fiGzZTKfHqLLLbK2e6Uop9/full_classification_dataset.csv"

try:
    print("Datensatz aus der Cloud laden...")
    df = pd.read_csv(cloud_url)
    if "Unnamed: 0" in df.columns:
        df = df.drop(columns=["Unnamed: 0"])
    print("Datensatz erfolgreich geladen!\n")
except Exception as e:
    print(f"Ein Fehler ist aufgetreten: {e}")

def combine_recipe_features(row):
    title = str(row['title']).strip()
    ingredients = str(row['NER']).replace('[', '').replace(']', '').replace("'", "").replace('"', '').strip()
    directions = str(row['directions']).replace('[', '').replace(']', '').replace("'", "").replace('"', '').strip()
    return f"Recipe: {title}\nIngredients: {ingredients}\nDirections: {directions}"

print("Bereite Datensatz für das Classification-Fine-Tuning vor...")

df['full_recipe'] = df.apply(combine_recipe_features, axis=1)
df = df[['full_recipe', 'genre', 'label']]

print("Aufbereitung erfolgreich abgeschlossen!\n")
df

Datensatz aus der Cloud laden...
Datensatz erfolgreich geladen!

Bereite Datensatz für das Classification-Fine-Tuning vor...
Aufbereitung erfolgreich abgeschlossen!



,full_recipe,genre,label
0,Recipe: Reeses Cups(Candy)\nIngredients: peanu...,drinks,2
1,Recipe: Rhubarb Coffee Cake\nIngredients: suga...,drinks,2
2,Recipe: Quick Barbecue Wings\nIngredients: chi...,nonveg,3
3,Recipe: Chocolate Frango Mints\nIngredients: c...,drinks,2
4,Recipe: Corral Barbecued Beef Steak Strips\nIn...,drinks,2
...,...,...,...
99995,Recipe: Basic Marinated And Baked Tofu\nIngred...,nonveg,3
99996,Recipe: Bouranee Baunjan - Afghan Eggplant Wit...,sides,8
99997,Recipe: Ginger-Soy-Lime Marinated Shrimp\nIngr...,drinks,2
99998,Recipe: Coffee Can Pumpkin Bread\nIngredients:...,cereal,6


In [47]:
genre_mapping = dict(df[['label', 'genre']].drop_duplicates().values)

In [51]:
print(genre_mapping)

{6: 'cereal', 1: 'bakery', 9: 'fusion', 4: 'vegetables', 7: 'meal', 8: 'sides', 2: 'drinks', 3: 'nonveg', 5: 'fastfood'}


In [17]:
# Anzeige der Anzahl der Rezepte pro Genre und Label
summary_df = df.groupby(['label', 'genre']).size().reset_index(name='count').sort_values('label')
print(summary_df.to_string(index=False))

 label      genre  count
     1     bakery   4112
     2     drinks  21293
     3     nonveg  14714
     4 vegetables  20200
     5   fastfood   6124
     6     cereal  16803
     7       meal   1687
     8      sides  11832
     9     fusion   3235


In [22]:
# Rezeptlänge in Tokens berechnen und filtern auf 1024, da in einem späteren Schritt das GPT-2 Modell eine maximale Kontextlänge von 1024 Tokens hat
df["token_count"] = [len(tokenizer.encode(text)) for text in df["full_recipe"]]
filtered_df = df[df["token_count"] <= 1024].drop(columns=["token_count"])

In [23]:
# Anzeige der Anzahl der Rezepte pro Genre und Label
summary_filtered_df = filtered_df.groupby(['label', 'genre']).size().reset_index(name='count').sort_values('label')
print(summary_filtered_df.to_string(index=False))

 label      genre  count
     1     bakery   4105
     2     drinks  21284
     3     nonveg  14707
     4 vegetables  20196
     5   fastfood   6118
     6     cereal  16789
     7       meal   1685
     8      sides  11826
     9     fusion   3230


In [24]:
# Undersampling um ein gleiches Verhältnis der Klassen zu erreichen
def create_balanced_dataset(df):
    min_size = df['label'].value_counts().min()
    
    subsets = []
    for label in df['label'].unique():
        label_subset = df[df['label'] == label].sample(min_size, random_state=123)
        subsets.append(label_subset)
        
    balanced_df = pd.concat(subsets).reset_index(drop=True)
    return balanced_df

balanced_df = create_balanced_dataset(filtered_df)
print(balanced_df.groupby(['label', 'genre']).size().reset_index(name='count').to_string(index=False))

 label      genre  count
     1     bakery   1685
     2     drinks   1685
     3     nonveg   1685
     4 vegetables   1685
     5   fastfood   1685
     6     cereal   1685
     7       meal   1685
     8      sides   1685
     9     fusion   1685


In [25]:
# Aufteilen in Trainings-, Validierungs- und Testset
def random_split(df, train_frac, validation_frac):
    df = df.sample(frac=1, random_state=123).reset_index(drop=True)

    train_end = int(len(df) * train_frac)
    validation_end = train_end + int(len(df) * validation_frac)

    train_df = df[:train_end]
    validation_df = df[train_end:validation_end]
    test_df = df[validation_end:]

    return train_df, validation_df, test_df

train_df, validation_df, test_df = random_split(balanced_df, 0.7, 0.1)

os.makedirs("classification", exist_ok=True)

train_df.to_csv("classification/train.csv", index=None)
validation_df.to_csv("classification/validation.csv", index=None)
test_df.to_csv("classification/test.csv", index=None)

## Erstellen eines spezifischen Dataloader

Da die Rezepte unterschiedlich lang sind, müssen wir sie für das Batch-Training vereinheitlichen. Dafür füllen wie allte Texte auf die länge des längsten Textes auf (Padding). Als Padding-Token nutzen wir das <|endoftext|> Token.

Training, Validierung und Testdatensatz padden.
- Note that validation and test set samples that are longer than the longest training example are being truncated via `encoded_text[:self.max_length]` in the `SpamDataset` code
- This behavior is entirely optional, and it would also work well if we set `max_length=None` in both the validation and test set cases

In [26]:
class RecipeDataset(Dataset):
    def __init__(self, csv_file, tokenizer, max_length=None, pad_token_id=50256):
        self.data = pd.read_csv(csv_file)

        self.encoded_texts = [
            tokenizer.encode(text) for text in self.data["full_recipe"]
        ]

        if max_length is None:
            self.max_length = self._longest_encoded_length()
        else:
            self.max_length = max_length
            self.encoded_texts = [
                encoded_text[:self.max_length]
                for encoded_text in self.encoded_texts
            ]

        # Pad sequences to the longest sequence
        self.encoded_texts = [
            encoded_text + [pad_token_id] * (self.max_length - len(encoded_text))
            for encoded_text in self.encoded_texts
        ]

    def __getitem__(self, index):
        encoded = self.encoded_texts[index]
        label = self.data.iloc[index]["label"]
        return (
            torch.tensor(encoded, dtype=torch.long),
            torch.tensor(label, dtype=torch.long)
        )

    def __len__(self):
        return len(self.data)

    def _longest_encoded_length(self):
        max_length = 0
        for encoded_text in self.encoded_texts:
            encoded_length = len(encoded_text)
            if encoded_length > max_length:
                max_length = encoded_length
        return max_length

    def _shortest_encoded_length(self):
        if not self.encoded_texts:
            return 0
        min_length = float('inf')
        for encoded_text in self.encoded_texts:
            encoded_length = len(encoded_text)
            if encoded_length < min_length:
                min_length = encoded_length
        return min_length

In [27]:
files = {
    "Train": "classification/train.csv",
    "Validation": "classification/validation.csv",
    "Test": "classification/test.csv"
}

for name, path in files.items():
    df = pd.read_csv(path)
    lengths = [len(tokenizer.encode(text)) for text in df["full_recipe"]]
    
    # Text vorab zusammenbauen
    min_str = f"{min(lengths)} Token"
    max_str = f"{max(lengths)} Token"
    
    # Jetzt den kompletten Text blockweise ausrichten
    print(f"{name:<10} | Kürzestes Rezept: {min_str:<4} | Längstes Rezept: {max_str}")

Train      | Kürzestes Rezept: 22 Token | Längstes Rezept: 982 Token
Validation | Kürzestes Rezept: 28 Token | Längstes Rezept: 833 Token
Test       | Kürzestes Rezept: 30 Token | Längstes Rezept: 879 Token


In [28]:
train_dataset = RecipeDataset(
    csv_file="classification/train.csv",
    max_length=None,
    tokenizer=tokenizer
)

print(train_dataset.max_length)

# Schneide die Texte im Validierungs- und Testset auf die maximale Länge des Trainingssets zu
val_dataset = RecipeDataset(
    csv_file="classification/validation.csv",
    max_length=train_dataset.max_length,
    tokenizer=tokenizer
)

print(val_dataset.max_length)

test_dataset = RecipeDataset(
    csv_file="classification/test.csv",
    max_length=train_dataset.max_length,
    tokenizer=tokenizer
)

print(test_dataset.max_length)

982
982
982


In [29]:
num_workers = 0
batch_size = 8

train_loader = DataLoader(
    dataset=train_dataset,
    batch_size=batch_size,
    shuffle=True,
    num_workers=num_workers,
    drop_last=True,
)

val_loader = DataLoader(
    dataset=val_dataset,
    batch_size=batch_size,
    num_workers=num_workers,
    drop_last=False,
)

test_loader = DataLoader(
    dataset=test_dataset,
    batch_size=batch_size,
    num_workers=num_workers,
    drop_last=False,
)

In [30]:
# verifiziere die Daten in einem Batch
print("Train loader:")
for input_batch, target_batch in train_loader:
    pass

print("Input batch dimensions:", input_batch.shape)
print("Label batch dimensions", target_batch.shape)

Train loader:
Input batch dimensions: torch.Size([8, 982])
Label batch dimensions torch.Size([8])


In [31]:
# Anzahl der Batches in jedem Loader
print(f"{len(train_loader)} training batches")
print(f"{len(val_loader)} validation batches")
print(f"{len(test_loader)} test batches")

1326 training batches
190 validation batches
380 test batches


## Load pre trained model

In [32]:
CHOOSE_MODEL = "gpt2-small (124M)"
INPUT_PROMPT = "Every effort moves"

BASE_CONFIG = {
    "vocab_size": 50257,     # Vocabulary size
    "context_length": 1024,  # Context length !!!!
    "drop_rate": 0.0,        # Dropout rate
    "qkv_bias": True         # Query-key-value bias
}

model_configs = {
    "gpt2-small (124M)": {"emb_dim": 768, "n_layers": 12, "n_heads": 12},
}

BASE_CONFIG.update(model_configs[CHOOSE_MODEL])

assert train_dataset.max_length <= BASE_CONFIG["context_length"], (
    f"Dataset length {train_dataset.max_length} exceeds model's context "
    f"length {BASE_CONFIG['context_length']}. Reinitialize data sets with "
    f"`max_length={BASE_CONFIG['context_length']}`"
)

In [ ]:
model_size = CHOOSE_MODEL.split(" ")[-1].lstrip("(").rstrip(")")
settings, params = download_and_load_gpt2(model_size=model_size, models_dir="gpt2")

model = GPTModel(BASE_CONFIG)
load_weights_into_gpt(model, params)
model.eval();

File already exists and is up-to-date: gpt2\124M\checkpoint
File already exists and is up-to-date: gpt2\124M\encoder.json
File already exists and is up-to-date: gpt2\124M\hparams.json
File already exists and is up-to-date: gpt2\124M\model.ckpt.data-00000-of-00001
File already exists and is up-to-date: gpt2\124M\model.ckpt.index
File already exists and is up-to-date: gpt2\124M\model.ckpt.meta
File already exists and is up-to-date: gpt2\124M\vocab.bpe


In [36]:
text_1 = "Recepie: Jewell Ball'S Chicken"

token_ids = generate_text_simple(
    model=model,
    idx=text_to_token_ids(text_1, tokenizer),
    max_new_tokens=15,
    context_size=BASE_CONFIG["context_length"]
)

print(token_ids_to_text(token_ids, tokenizer))

Recepie: Jewell Ball'S Chicken

[00:00:00]EMOTE: *no key*/(


In [38]:
text_2 = (
    "What would you label the following recipe? Answer with 'drinks', 'nonveg', 'vegetables', 'fastfood', 'cereal', 'meal', 'sides', or 'fusion':"
    " Recipe: Meyer Lemon Marmalade\nIngredients: lemons, sugar\nDirections: Rinse the lemons and pat dry. Halve the lemons crosswise and juice them, reserving the juice. Using a spoon, scrape the pulp and seeds from the halves. Using a sharp knife, slice the peels 1/8 inch thick., In a large, heavy saucepan, cover the strips with 8 cups of cold water and bring to a boil; boil for 1 minute. Drain the strips and rinse under cold running water. Blanch two more times; the final time, drain the strips but do not rinse them., Return the strips to the saucepan. Add the reserved juice and the sugar. Simmer over moderate heat, stirring to dissolve the sugar, then skimming any foam, until the marmalade sets, about 30 minutes., Spoon the marmalade into 5 hot 1/2-pint canning jars, leaving 1/4 inch of space at the top, and close with the lids and rings. To process, boil the jars for 15 minutes in water to cover. Let stand at room temperature for 2 days before serving., MAKE AHEAD The processed marmalade can be stored in a cool, dark place for up to 1 year. Refrigerate after opening."
)

token_ids = generate_text_simple(
    model=model,
    idx=text_to_token_ids(text_2, tokenizer),
    max_new_tokens=23,
    context_size=BASE_CONFIG["context_length"]
)

print(token_ids_to_text(token_ids, tokenizer))

What would you label the following recipe? Answer with 'drinks', 'nonveg', 'vegetables', 'fastfood', 'cereal', 'meal', 'sides', or 'fusion': Recipe: Meyer Lemon Marmalade
Ingredients: lemons, sugar
Directions: Rinse the lemons and pat dry. Halve the lemons crosswise and juice them, reserving the juice. Using a spoon, scrape the pulp and seeds from the halves. Using a sharp knife, slice the peels 1/8 inch thick., In a large, heavy saucepan, cover the strips with 8 cups of cold water and bring to a boil; boil for 1 minute. Drain the strips and rinse under cold running water. Blanch two more times; the final time, drain the strips but do not rinse them., Return the strips to the saucepan. Add the reserved juice and the sugar. Simmer over moderate heat, stirring to dissolve the sugar, then skimming any foam, until the marmalade sets, about 30 minutes., Spoon the marmalade into 5 hot 1/2-pint canning jars, leaving 1/4 inch of space at the top, and close with the lids and rings. To process, 

In [37]:
balanced_df["full_recipe"].iloc[0]

'Recipe: Meyer Lemon Marmalade\nIngredients: lemons, sugar\nDirections: Rinse the lemons and pat dry. Halve the lemons crosswise and juice them, reserving the juice. Using a spoon, scrape the pulp and seeds from the halves. Using a sharp knife, slice the peels 1/8 inch thick., In a large, heavy saucepan, cover the strips with 8 cups of cold water and bring to a boil; boil for 1 minute. Drain the strips and rinse under cold running water. Blanch two more times; the final time, drain the strips but do not rinse them., Return the strips to the saucepan. Add the reserved juice and the sugar. Simmer over moderate heat, stirring to dissolve the sugar, then skimming any foam, until the marmalade sets, about 30 minutes., Spoon the marmalade into 5 hot 1/2-pint canning jars, leaving 1/4 inch of space at the top, and close with the lids and rings. To process, boil the jars for 15 minutes in water to cover. Let stand at room temperature for 2 days before serving., MAKE AHEAD The processed marmala

In [39]:
# GPT-2 Gewichte einfrieren, damit sie während des Fine-Tunings nicht aktualisiert werden
for param in model.parameters():
    param.requires_grad = False

In [41]:
num_classes = 9
model.out_head = torch.nn.Linear(in_features=BASE_CONFIG["emb_dim"], out_features=num_classes)

for param in model.trf_blocks[-1].parameters():
    param.requires_grad = True

for param in model.final_norm.parameters():
    param.requires_grad = True

In [ ]:
inputs = tokenizer.encode("Recepie: Jewell Ball'S Chicken")
inputs = torch.tensor(inputs).unsqueeze(0)
print("Inputs:", inputs)
print("Inputs dimensions:", inputs.shape)

Inputs: tensor([[ 3041,   344, 21749,    25,  3370,   695,  6932,     6,    50, 16405]])
Inputs dimensions: torch.Size([1, 10])


In [44]:
with torch.no_grad():
    outputs = model(inputs)

print("Outputs:\n", outputs)
print("Outputs dimensions:", outputs.shape) # shape: (batch_size, num_tokens, num_classes)

Outputs:
 tensor([[[-1.1331,  1.1905, -0.1248,  0.1773,  0.2151, -2.4889,  1.3837,
          -1.1788,  0.0354],
         [-3.0206,  2.1569, -0.0320,  1.9674,  0.8175, -5.4105,  3.0843,
          -3.0343, -1.3717],
         [-4.5329,  2.7589,  0.1050,  1.8113,  0.9763, -6.4313,  4.1716,
          -4.4195, -1.5107],
         [-6.9967,  3.1498, -0.0491,  4.2395,  1.2333, -8.1258,  6.1207,
          -6.1674, -3.2375],
         [-5.1332,  2.2683,  0.5170,  1.9831,  0.7794, -6.6847,  4.0684,
          -4.6816, -1.6671],
         [-4.2867,  1.7577,  0.7294,  1.9375,  0.6529, -6.1746,  3.5883,
          -2.9873, -0.8464],
         [-4.9299,  2.4816,  0.7396,  1.8206,  0.7730, -6.3684,  3.8475,
          -3.8969, -1.4986],
         [-4.5428,  1.9792,  0.4017,  1.4959,  0.8311, -6.3469,  3.9341,
          -3.9991, -1.7234],
         [-5.3671,  2.2752,  1.1394,  2.8094,  0.7236, -6.5337,  3.9170,
          -4.4028, -2.3601],
         [-5.4216,  3.4527, -0.1145,  1.3061,  0.8960, -7.3946,  4.3571,

## Output anpassen auf Multi Label approach

In [53]:
import torch

# 1. Logits durch Sigmoid jagen und flachmachen
probas = torch.sigmoid(outputs[:, -1, :])[0].tolist()

# 2. Direkt die um +1 verschobenen IDs sammeln, die über dem Threshold liegen (1 bis 9)
active_ids = [i + 1 for i, p in enumerate(probas) if p > 0.5]

# 3. Die Genres direkt über die korrigierten IDs abfragen
active_genres = [genre_mapping[idx] for idx in active_ids]

# Reine Listen-Ausgabe
print(probas)
print(active_ids)
print(active_genres)

[0.004400409758090973, 0.969311535358429, 0.4714137613773346, 0.7868637442588806, 0.7101225852966309, 0.0006141739431768656, 0.987346351146698, 0.008930004201829433, 0.1755446344614029]
[2, 4, 5, 7]
['drinks', 'vegetables', 'fastfood', 'meal']


In [ ]:
def calc_accuracy_loader(data_loader, model, device, num_batches=None):
    model.eval()
    correct_predictions, num_examples = 0, 0

    if num_batches is None:
        num_batches = len(data_loader)
    else:
        num_batches = min(num_batches, len(data_loader))
    for i, (input_batch, target_batch) in enumerate(data_loader):
        if i < num_batches:
            input_batch, target_batch = input_batch.to(device), target_batch.to(device)

            with torch.no_grad():
                logits = model(input_batch)[:, -1, :]  # Logits of last output token
            predicted_labels = torch.argmax(logits, dim=-1)

            num_examples += predicted_labels.shape[0]
            correct_predictions += (predicted_labels == target_batch).sum().item()
        else:
            break
    return correct_predictions / num_examples